# PySpark: от ноутбука до истории выполнения

**Jupyter → Spark Driver → Master → Worker/Executor → event log → History Server.**

Учебный кластер работает на одной VM GitHub Codespaces. Выполняйте ячейки по очереди: **Shift+Enter**.

Порты: **8888** — ноутбук, **8080** — Master, **4040** — текущее приложение, **18080** — история.

## 1. Подключаемся к настоящему worker

Driver живёт в контейнере ноутбука; вычисления выполняет executor в контейнере worker. Это `spark://spark-master:7077`, а не `local[*]`.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("Полка — аналитика 1 000 000 заказов")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)
print("Spark UI: откройте порт 4040 через Ports")

## 2. Создаём миллион учебных заказов

Данные генерируются в Spark без внешних файлов. Восемь исходных разделов, четыре города, сумма заказа — целое число копеек: точность не теряется.

In [ ]:
cities = F.array(*[F.lit(c) for c in ["Москва", "Санкт-Петербург", "Казань", "Екатеринбург"]])
orders = (spark.range(1_000_000, numPartitions=8)
          .select(F.col("id").alias("order_id"),
                  cities[(F.col("id") % 4).cast("int")].alias("city"),
                  ((F.col("id") * 37) % 10_000 + 100).alias("amount_kopecks")))
print("Исходных разделов:", orders.rdd.getNumPartitions())
orders.show(8, truncate=False)

## 3. Считаем выручку по городам

`groupBy` создаёт обмен данными — **shuffle**. Spark ленивый: вычисление запускается при действии `show()`. В **4040 → Jobs / Stages / SQL** смотрим задачи и физический план. AQE может объединить небольшие разделы.

In [ ]:
summary = (orders.groupBy("city")
           .agg(F.count("*").alias("orders"),
                F.sum("amount_kopecks").alias("revenue_kopecks"))
           .orderBy(F.desc("revenue_kopecks")))
summary.show(truncate=False)
assert summary.agg(F.sum("orders")).first()[0] == 1_000_000
print("Проверено: всего 1 000 000 заказов")

## 4. Сохраняем результат в Parquet

Spark выполняет ещё одно действие — запись. Результат хранится в отдельном Docker volume и переживает остановку контейнеров.

In [ ]:
summary.write.mode("overwrite").parquet("/opt/work/results/orders_by_city")
spark.read.parquet("/opt/work/results/orders_by_city").show(truncate=False)
print("Результат сохранён. Теперь посмотрите Spark UI на порте 4040.")

## 5. Завершаем приложение и открываем History Server

После `spark.stop()` текущий Spark UI на **4040** перестаёт работать. Но event log сохранён: через несколько секунд приложение появится на **18080** как завершённое. Там доступны те же Jobs, Stages, SQL и Executors.

In [ ]:
app_id = spark.sparkContext.applicationId
spark.stop()
print("Приложение завершено:", app_id)
print("Откройте History Server: порт 18080 → название приложения")

## Закончили занятие

В терминале Codespace: `make -C spark-lab down`. Затем **Ctrl+Shift+P → Codespaces: Stop Current Codespace**.

Для продолжения: откройте тот же Codespace, выполните `make -C spark-lab up` и снова первую ячейку создания SparkSession. Не удаляйте тома: в них история и результаты.